# Lernmaterial-Generator – LangGraph-Agent für Notebook-Quizze

Dieses Notebook ist **selbst Lerngegenstand**: Es zeigt anhand eines praktischen
Anwendungsfalls, wie sich mit **LangGraph** ein Agenten-Workflow mit
*Reflexions-Schleife* aufbauen lässt.

**Was der Agent tut** – aus zwei Eingaben
1. einem hochgeladenen **Jupyter-Notebook** (Code *und* Markdown werden analysiert) und
2. einem **Schwerpunkt-Text** des Dozenten

erzeugt er drei aufeinander abgestimmte Artefakte:

1. ein **knappes, didaktisch sauberes Markdown-Skript** mit dem nötigen Theorie-Hintergrund,
2. eine **Liste der zugehörigen Fachtermini** und
3. eine **eigenständige HTML-Quiz-Seite** mit 10 Fragen im gewohnten Layout.

Ein **Qualitäts-/Reflexions-Knoten** prüft das Ergebnis (Halluzinationen, Wortzahl-Balance der
Antworten, Eindeutigkeit der Lösung, Schwerpunkt-Abdeckung) und löst bei Bedarf eine
Überarbeitung aus – **maximal drei Iterationen**.

Studierende können beliebig oft **neue Quizze** erzeugen (mit Schwierigkeitsstufe 1–5);
eine Anti-Dubletten-Logik verhindert Wiederholungen.

> **Modelle:** provider-agnostisch über die OpenAI-kompatible Schnittstelle –
> funktioniert mit **LM Studio** (lokal), **DeepInfra** und **OpenRouter**.
> Für die Generierung genügen drei Felder (`base_url`, `api_key`, `model`); für den
> Qualitäts-Checker lässt sich optional ein **eigenes (stärkeres) Modell** angeben.


In [ ]:
# Einmalig ausführen, falls die Pakete fehlen:
# %pip install langgraph langchain-core openai gradio nbformat

In [ ]:
import json
import re
import random
import html as html_lib
import tempfile
from pathlib import Path
from typing import TypedDict, List, Dict, Optional

import nbformat
from openai import OpenAI

import gradio as gr
from langgraph.graph import StateGraph, END

## Schritt 1 – Provider-agnostischer LLM-Client

LM Studio, DeepInfra und OpenRouter sprechen alle die **OpenAI-kompatible** API.
Deshalb genügt ein einziger Client, der über `base_url` umgeschaltet wird.

`call_llm()` ist die zentrale Aufruf-Funktion aller Graph-Knoten. Die Konfiguration
(`base_url`, `api_key`, `model`) reist im **State** mit – so bleiben die Knoten zustandslos
und gut testbar. Ein robuster JSON-Extraktor fängt typische Modell-Macken ab
(Code-Fences, Vor-/Nachtext).

In [ ]:
# Globaler Token-Zähler. Wird zu Beginn jedes Laufs zurückgesetzt und in call_llm
# nach jeder Antwort fortgeschrieben. Ausgelegt für den Einzelnutzer-Betrieb (eine
# Gradio-Sitzung); bei echtem Mehrnutzer-Parallelbetrieb müsste man ihn pro Sitzung halten.
TOKENS = {"prompt": 0, "completion": 0, "total": 0, "calls": 0}

def reset_tokens():
    TOKENS.update(prompt=0, completion=0, total=0, calls=0)

def _add_usage(resp):
    u = getattr(resp, "usage", None)
    if not u:
        return  # manche Endpunkte liefern keine usage-Daten
    TOKENS["prompt"]     += getattr(u, "prompt_tokens", 0) or 0
    TOKENS["completion"] += getattr(u, "completion_tokens", 0) or 0
    TOKENS["total"]      += getattr(u, "total_tokens", 0) or 0
    TOKENS["calls"]      += 1


def call_llm(state: dict, system: str, user: str,
             temperature: float = 0.3, max_tokens: int = 4000,
             role: str = "generator") -> str:
    """Ein Chat-Completion-Aufruf gegen einen OpenAI-kompatiblen Endpunkt.

    role="checker" nutzt – falls gesetzt – ein eigenes Checker-Modell. Bleiben die
    Checker-Felder leer, wird automatisch das Generator-Modell verwendet. base_url und
    api_key des Checkers sind ebenfalls optional und fallen sonst auf die Generator-Werte
    zurück (so kann der Checker auch bei einem anderen Provider liegen).
    """
    if role == "checker" and state.get("checker_model"):
        base_url = (state.get("checker_base_url") or state["base_url"]).rstrip("/")
        api_key = state.get("checker_api_key") or state.get("api_key") or "not-needed"
        model = state["checker_model"]
    else:
        base_url = state["base_url"].rstrip("/")
        api_key = state.get("api_key") or "not-needed"   # LM Studio braucht keinen echten Key
        model = state["model"]
    client = OpenAI(base_url=base_url, api_key=api_key)
    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    _add_usage(resp)
    return resp.choices[0].message.content or ""


def extract_json(text: str):
    """Holt das erste gültige JSON-Array/-Objekt aus einer Modellantwort."""
    t = text.strip()
    t = re.sub(r"^```(?:json)?", "", t).strip()
    t = re.sub(r"```$", "", t).strip()
    for opener, closer in (("[", "]"), ("{", "}")):
        start, end = t.find(opener), t.rfind(closer)
        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(t[start:end + 1])
            except json.JSONDecodeError:
                continue
    raise ValueError("Konnte kein gültiges JSON aus der Modellantwort lesen.")

## Schritt 2 – Notebook-Analyse

Wir lesen mit `nbformat` **Code- und Markdown-Zellen** getrennt aus. Beides bildet
die *Faktenbasis* (das »Grounding«), auf die sich Skript und Quiz stützen dürfen.

In [ ]:
def analyze_notebook(path: str) -> Dict[str, str]:
    """Trennt Code- und Markdown-Inhalte eines .ipynb."""
    nb = nbformat.read(path, as_version=4)
    code_parts, md_parts = [], []
    for cell in nb.cells:
        src = (cell.get("source") or "").strip()
        if not src:
            continue
        if cell.cell_type == "code":
            code_parts.append(src)
        elif cell.cell_type == "markdown":
            md_parts.append(src)
    return {
        "notebook_code": "\n\n# --- nächste Code-Zelle ---\n".join(code_parts),
        "notebook_markdown": "\n\n---\n".join(md_parts) if md_parts else "(keine Markdown-Zellen vorhanden)",
    }

## Schritt 3 – HTML-Template (eigenständig, ohne Server lauffähig)

Das gewohnte Layout bleibt eine **feste Hülle**. Das Sprachmodell erzeugt ausschließlich
das `questions`-Array; wir spritzen es **inline** in die Seite ein.

Das ist bewusst so gelöst: Würde man die Fragen aus einer separaten `.json` per `fetch()`
nachladen, blockiert der Browser das beim Öffnen über `file://` (CORS / Local-File-Policy).
**Inline-Einbettung** umgeht das – die fertige `.html` lässt sich per Doppelklick von der
Festplatte öffnen, ganz ohne lokalen Server.

In [ ]:
QUIZ_TEMPLATE = r"""<!DOCTYPE html>
<html lang="de">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>__QUIZ_TITLE__</title>
    <style>
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background-color: #f4f4f9;
            margin: 0;
            padding: 0;
            color: #333;
        }
        .status-bar {
            position: sticky;
            top: 0;
            background: white;
            padding: 15px 20px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
            z-index: 100;
            display: flex;
            flex-direction: column;
            gap: 10px;
        }
        .status-info {
            display: flex;
            justify-content: space-between;
            font-weight: bold;
            font-size: 1.1rem;
        }
        .mistakes { color: #dc3545; }
        .progress-container {
            width: 100%;
            background-color: #e0e0e0;
            border-radius: 8px;
            overflow: hidden;
            height: 12px;
        }
        .progress-bar {
            height: 100%;
            background-color: #28a745;
            width: 0%;
            transition: width 0.4s ease;
        }
        .quiz-container {
            max-width: 800px;
            margin: 30px auto;
            padding: 0 20px;
            display: flex;
            flex-direction: column;
            gap: 30px;
        }
        .quiz-card {
            background: white;
            padding: 2rem;
            border-radius: 12px;
            box-shadow: 0 4px 15px rgba(0,0,0,0.05);
        }
        h1 { text-align: center; color: #2c3e50; margin-top: 40px; }
        .question-title {
            color: #7f8c8d;
            font-size: 0.9rem;
            margin-bottom: 5px;
            text-transform: uppercase;
            letter-spacing: 1px;
        }
        .question-text {
            font-size: 1.2rem;
            color: #2c3e50;
            margin-bottom: 20px;
            font-weight: 600;
        }
        .options { display: flex; flex-direction: column; gap: 10px; }
        button {
            padding: 12px 15px;
            border: 2px solid #e0e0e0;
            background: white;
            border-radius: 8px;
            cursor: pointer;
            font-size: 1rem;
            transition: all 0.2s;
            text-align: left;
        }
        button:hover:not([disabled]) {
            background-color: #f0f8ff;
            border-color: #3498db;
        }
        button:disabled { cursor: not-allowed; opacity: 0.7; }
        button.correct {
            background-color: #d4edda !important;
            border-color: #28a745 !important;
            color: #155724;
        }
        button.wrong {
            background-color: #f8d7da !important;
            border-color: #dc3545 !important;
            color: #721c24;
            text-decoration: line-through;
        }
        .feedback {
            margin-top: 15px;
            padding: 15px;
            border-radius: 8px;
            display: none;
            font-size: 0.95rem;
            line-height: 1.5;
        }
        .feedback.correct-feedback {
            background-color: #d4edda;
            color: #155724;
            border: 1px solid #c3e6cb;
        }
        .feedback.wrong-feedback {
            background-color: #f8d7da;
            color: #721c24;
            border: 1px solid #f5c6cb;
        }
        .final-result {
            display: none;
            text-align: center;
            background: #e8f4fd;
            border: 2px solid #3498db;
            padding: 2rem;
            border-radius: 12px;
            margin-bottom: 40px;
        }
    </style>
</head>
<body>

<div class="status-bar">
    <div class="status-info">
        <span>Fortschritt: <span id="progress-text">0%</span></span>
        <span class="mistakes">Fehler: <span id="mistake-counter">0</span></span>
    </div>
    <div class="progress-container">
        <div class="progress-bar" id="progress-bar"></div>
    </div>
</div>

<h1>__QUIZ_TITLE__</h1>

<div class="quiz-container" id="quiz-container"></div>

<div class="quiz-container">
    <div class="final-result" id="final-result">
        <h2>Quiz abgeschlossen!</h2>
        <p>Du hast alle Fragen beantwortet. Insgesamt hast du <strong id="final-mistakes">0</strong> Fehler gemacht.</p>
        <button onclick="location.reload()" style="background: #3498db; color: white; border: none; text-align: center; margin-top: 15px; padding: 12px 24px; font-size: 1.1rem; border-radius: 8px; cursor: pointer;">Quiz neustarten</button>
    </div>
</div>

<script>
    const questions = /*__QUESTIONS__*/[];

    let correctCount = 0;
    let mistakeCount = 0;

    function initQuiz() {
        const container = document.getElementById('quiz-container');
        questions.forEach((q, qIndex) => {
            const card = document.createElement('div');
            card.className = 'quiz-card';
            card.id = `question-card-${qIndex}`;
            const title = document.createElement('div');
            title.className = 'question-title';
            title.textContent = `Frage ${qIndex + 1} von ${questions.length}`;
            const text = document.createElement('div');
            text.className = 'question-text';
            text.textContent = q.text;
            const optionsContainer = document.createElement('div');
            optionsContainer.className = 'options';
            q.options.forEach((opt, optIndex) => {
                const btn = document.createElement('button');
                btn.textContent = opt;
                btn.onclick = () => handleAnswer(qIndex, optIndex, btn);
                optionsContainer.appendChild(btn);
            });
            const feedback = document.createElement('div');
            feedback.className = 'feedback';
            feedback.id = `feedback-${qIndex}`;
            card.appendChild(title);
            card.appendChild(text);
            card.appendChild(optionsContainer);
            card.appendChild(feedback);
            container.appendChild(card);
        });
    }

    function handleAnswer(qIndex, optIndex, btnElement) {
        const q = questions[qIndex];
        const card = document.getElementById(`question-card-${qIndex}`);
        const feedback = document.getElementById(`feedback-${qIndex}`);
        if (card.classList.contains('completed')) return;
        if (optIndex === q.correct) {
            btnElement.classList.add('correct');
            card.classList.add('completed');
            const allBtns = card.querySelectorAll('button');
            allBtns.forEach(b => b.disabled = true);
            feedback.className = 'feedback correct-feedback';
            feedback.innerHTML = `<strong>Richtig!</strong> ${q.explanation}`;
            feedback.style.display = 'block';
            correctCount++;
            updateProgress();
        } else {
            btnElement.classList.add('wrong');
            btnElement.disabled = true;
            mistakeCount++;
            document.getElementById('mistake-counter').textContent = mistakeCount;
            feedback.className = 'feedback wrong-feedback';
            feedback.innerHTML = `<strong>Leider falsch.</strong> Versuche es noch einmal!`;
            feedback.style.display = 'block';
        }
    }

    function updateProgress() {
        const percent = Math.round((correctCount / questions.length) * 100);
        document.getElementById('progress-bar').style.width = percent + '%';
        document.getElementById('progress-text').textContent = percent + '%';
        if (correctCount === questions.length) {
            document.getElementById('final-mistakes').textContent = mistakeCount;
            document.getElementById('final-result').style.display = 'block';
        }
    }

    initQuiz();
</script>

</body>
</html>
"""

In [ ]:
def build_quiz_html(questions: List[dict], title: str) -> str:
    """Injiziert das Fragen-Array (als JSON, das gültiges JS ist) und den Titel."""
    q_json = json.dumps(questions, ensure_ascii=False, indent=8)
    out = QUIZ_TEMPLATE.replace("__QUIZ_TITLE__", title or "Wissens-Check")
    out = out.replace("/*__QUESTIONS__*/[]", q_json)
    return out

## Schritt 4 – Die Prompts (der eigentliche »Skill«)

Hier sitzt die fachliche Intelligenz. Die Prompts sind **domänen-unabhängig** formuliert –
sie funktionieren für PyTorch ebenso wie für sklearn, pandas, seaborn, Keras/CNN,
Transfer-Learning, ARIMA/Prophet, gensim oder NLTK.

**Zentrale Leitplanken**

- **Grounding gegen Halluzinationen:** Es darf nur behauptet werden, was (a) durch den
  Notebook-Code belegt oder (b) gesichertes, allgemein anerkanntes Fachwissen ist.
  *Keine* erfundenen API-Signaturen, Parameter, Default-Werte oder Zahlen.
- **Bloom-Mischung nach Schwierigkeit (1–5):** je höher die Stufe, desto stärker das Gewicht
  hochwertiger Taxonomiestufen.
- **Wortzahl-Balance:** alle vier Optionen ähnlich lang und gleich strukturiert – die
  richtige Antwort darf **nicht** die längste/detaillierteste sein (typischer LLM-Tell).
- **Anti-Dubletten:** bereits gestellte Themen/Fragestämme werden explizit ausgeschlossen.

In [ ]:
def bloom_guidance(difficulty: int) -> str:
    """Verteilung der Bloom-Stufen je nach Schwierigkeit 1-5."""
    table = {
        1: "Schwerpunkt ERINNERN & VERSTEHEN (~70 %), wenig ANWENDEN (~30 %). Klare, einfache Distraktoren.",
        2: "VERSTEHEN überwiegt (~50 %), ANWENDEN (~35 %), erste ANALYSE-Fragen (~15 %).",
        3: "Ausgewogen: VERSTEHEN/ANWENDEN/ANALYSIEREN zu etwa gleichen Teilen. Distraktoren plausibel.",
        4: "Schwerpunkt ANWENDEN & ANALYSIEREN (~60 %), BEWERTEN (~25 %), wenig reines Erinnern. Feine Distraktoren.",
        5: "Hochwertige Stufen: ANALYSIEREN/BEWERTEN/ERSCHAFFEN (~75 %), Mehrschritt-Reasoning, sehr plausible, nah beieinanderliegende Distraktoren.",
    }
    d = max(1, min(5, int(difficulty)))
    return f"Schwierigkeitsstufe {d}/5 – {table[d]}"


GROUNDING_RULE = (
    "STRIKTES GROUNDING: Stütze dich ausschließlich auf (a) den bereitgestellten "
    "Notebook-Code samt Kommentaren und Markdown sowie (b) allgemein anerkanntes, "
    "gesichertes Fachwissen der dort verwendeten Bibliotheken. Erfinde NIEMALS "
    "API-Signaturen, Parameternamen, Default-Werte, Kennzahlen oder Funktionsverhalten. "
    "Wenn du dir nicht sicher bist, formuliere allgemeiner statt zu spekulieren."
)


def build_script_prompt(state: dict) -> tuple:
    system = (
        "Du bist eine erfahrene Dozentin für Machine Learning und erstellst knappe, "
        "didaktisch saubere Lern-Skripte auf Deutsch im Markdown-Format. " + GROUNDING_RULE
    )
    user = f"""Analysiere das folgende Jupyter-Notebook (Code und Markdown) und erstelle ein
**knappes** Markdown-Skript, das den nötigen theoretischen Hintergrund zum Verständnis des
Codes vermittelt. Nicht die Länge zählt, sondern didaktische Klarheit.

Beginne mit einer Überschrift der Form `# <kurzer, prägnanter Titel>`.
Erkläre die zentralen Konzepte, Fachbegriffe und das Warum hinter den Code-Schritten.
Gehe besonders auf die folgenden Schwerpunkte des Dozenten ein:
---SCHWERPUNKTE---
{state.get('focus') or '(keine besonderen Schwerpunkte angegeben)'}
---ENDE SCHWERPUNKTE---

NOTEBOOK-CODE:
{state['notebook_code']}

NOTEBOOK-MARKDOWN:
{state['notebook_markdown']}
"""
    if state.get("quality", {}).get("script_feedback"):
        user += f"\n\nUEBERARBEITUNG NÖTIG – berücksichtige diese Kritik:\n{state['quality']['script_feedback']}"
    return system, user


def build_termini_prompt(state: dict) -> tuple:
    system = ("Du extrahierst Fachtermini aus Lern-Skripten. Antworte AUSSCHLIESSLICH mit "
              "einem JSON-Array von Strings, ohne weiteren Text. " + GROUNDING_RULE)
    user = f"""Lies das folgende Skript und gib die wichtigsten Fachtermini als JSON-Array
von kurzen Strings zurück (je Begriff optional mit knapper Klärung in Klammern).
Nur Begriffe, die im Skript oder Notebook tatsächlich vorkommen.

SKRIPT:
{state.get('script_md','')}
"""
    return system, user


def build_quiz_prompt(state: dict) -> tuple:
    avoid = state.get("avoid_topics") or []
    avoid_txt = "\n".join(f"- {a}" for a in avoid) if avoid else "(noch keine)"
    spotlight = state.get("quiz_focus_termini") or []
    spotlight_block = ""
    if spotlight:
        spotlight_block = (
            "\n- SCHWERPUNKT DIESES DURCHLAUFS: Baue die Fragen GEZIELT rund um diese "
            "Fachbegriffe auf und beleuchte sie aus einem ANDEREN Blickwinkel als üblich:\n"
            + "\n".join(f"  · {t}" for t in spotlight)
        )
    system = (
        "Du bist Prüfungs-Didaktikerin und erstellst hochwertige Multiple-Choice-Fragen "
        "auf Deutsch. Antworte AUSSCHLIESSLICH mit einem JSON-Array, ohne weiteren Text. "
        + GROUNDING_RULE
    )
    user = f"""Erstelle GENAU 10 Multiple-Choice-Fragen zum folgenden Stoff.

{bloom_guidance(state.get('difficulty', 3))}

FORMAT – ein JSON-Array mit 10 Objekten, jeweils:
{{"text": "Frage?", "options": ["A","B","C","D"], "correct": <Index 0-3>, "explanation": "kurze Begründung"}}

HARTE REGELN:
- Genau 4 Optionen pro Frage, genau eine ist eindeutig korrekt.
- WORTZAHL-BALANCE: Alle vier Optionen müssen ähnlich lang und gleich strukturiert sein.
  Die korrekte Antwort darf NICHT systematisch die längste oder detaillierteste sein.
  Verteile die Position der korrekten Antwort gleichmäßig über die Indizes 0-3.
- Distraktoren müssen fachlich plausibel und verlockend sein, aber eindeutig falsch.
- Decke die Dozenten-Schwerpunkte ab: {state.get('focus') or '(keine)'}{spotlight_block}
- Vermeide thematische Wiederholungen zu bereits gestellten Fragen:
{avoid_txt}

SKRIPT (Grounding):
{state.get('script_md','')}

NOTEBOOK-CODE (Grounding):
{state['notebook_code']}
"""
    if state.get("quality", {}).get("quiz_feedback"):
        user += f"\n\nUEBERARBEITUNG NÖTIG – behebe diese Mängel:\n{state['quality']['quiz_feedback']}"
    return system, user


def build_checker_prompt(state: dict) -> tuple:
    system = (
        "Du bist ein strenger Qualitätsprüfer für Lernmaterial. Du prüfst faktische "
        "Korrektheit gegen die gegebene Faktenbasis und meldest jede nicht belegte Aussage. "
        "Antworte AUSSCHLIESSLICH mit einem JSON-Objekt."
    )
    user = f"""Prüfe Skript und Quiz gegen die Faktenbasis (Notebook + gesichertes Fachwissen).

Gib ein JSON-Objekt zurück:
{{"script_ok": bool, "quiz_ok": bool,
  "script_feedback": "konkrete Mängel oder ''",
  "quiz_feedback": "konkrete Mängel oder ''"}}

Prüfkriterien:
1. HALLUZINATIONEN: Werden API-Funktionen, Parameter, Default-Werte oder Zahlen behauptet,
   die weder im Notebook stehen noch gesichertes Allgemeinwissen sind? -> nicht ok.
2. Hat jede Quizfrage genau eine eindeutig korrekte Antwort?
3. Sind die Distraktoren fachlich plausibel (keine offensichtlich absurden Optionen)?
4. Werden die Dozenten-Schwerpunkte abgedeckt: {state.get('focus') or '(keine)'} ?
5. Passt das Anspruchsniveau zur Stufe {state.get('difficulty', 3)}/5?

NOTEBOOK-CODE:
{state['notebook_code']}

SKRIPT:
{state.get('script_md','')}

QUIZ (JSON):
{json.dumps(state.get('quiz_questions', []), ensure_ascii=False)}
"""
    return system, user

## Schritt 5 – Der gemeinsame Zustand (State)

In LangGraph reichen die Knoten einen gemeinsamen `State` weiter. Jeder Knoten gibt ein
**Teil-Dictionary** zurück, das in den State gemischt wird.

In [ ]:
class QuizState(TypedDict, total=False):
    # Konfiguration / Eingaben
    base_url: str
    api_key: str
    model: str
    checker_base_url: str
    checker_api_key: str
    checker_model: str
    notebook_code: str
    notebook_markdown: str
    focus: str
    difficulty: int
    avoid_topics: List[str]
    quiz_focus_termini: List[str]
    quiz_temperature: float
    # Zwischen- und Endergebnisse
    script_md: str
    title: str
    termini: List[str]
    quiz_questions: List[dict]
    quiz_html: str
    quality: dict
    iteration: int

## Schritt 6 – Die Graph-Knoten

`script -> termini -> quiz -> quality`. Der Qualitäts-Knoten kombiniert **programmatische
Checks** (Wortzahl-Balance, Struktur – objektiv und billig) mit einer **LLM-Bewertung**
(Halluzinationen, Plausibilität – semantisch).

In [ ]:
def node_script(state: QuizState) -> dict:
    system, user = build_script_prompt(state)
    script = call_llm(state, system, user, temperature=0.25)
    m = re.search(r"^#\s+(.+)$", script, flags=re.MULTILINE)
    title = m.group(1).strip() if m else "Wissens-Check"
    return {"script_md": script, "title": title}


def node_termini(state: QuizState) -> dict:
    system, user = build_termini_prompt(state)
    raw = call_llm(state, system, user, temperature=0.2)
    try:
        termini = extract_json(raw)
        if not isinstance(termini, list):
            termini = [str(termini)]
    except ValueError:
        termini = [line.strip("-* ").strip() for line in raw.splitlines() if line.strip()]
    return {"termini": [str(t) for t in termini]}


def node_quiz(state: QuizState) -> dict:
    system, user = build_quiz_prompt(state)
    temp = state.get("quiz_temperature", 0.7)
    questions = []
    for _ in range(2):  # ein interner Wiederholversuch bei JSON-Fehler
        raw = call_llm(state, system, user, temperature=temp)
        try:
            questions = extract_json(raw)
            if isinstance(questions, list) and questions:
                break
        except ValueError:
            continue
    return {"quiz_questions": questions, "iteration": state.get("iteration", 0) + 1}


def node_quality(state: QuizState) -> dict:
    # 1) Programmatische Checks
    prog_issues = structural_checks(state["quiz_questions"]) + word_balance_check(state["quiz_questions"])
    # 2) LLM-Bewertung (faktentreu, niedrige Temperatur)
    system, user = build_checker_prompt(state)
    try:
        verdict = extract_json(call_llm(state, system, user, temperature=0.1, role="checker"))
    except ValueError:
        verdict = {"script_ok": True, "quiz_ok": True, "script_feedback": "", "quiz_feedback": ""}

    quiz_fb = (verdict.get("quiz_feedback") or "").strip()
    if prog_issues:
        quiz_fb = (quiz_fb + " | " if quiz_fb else "") + " ".join(prog_issues)
    quiz_ok = bool(verdict.get("quiz_ok", True)) and not prog_issues
    return {"quality": {
        "script_ok": bool(verdict.get("script_ok", True)),
        "quiz_ok": quiz_ok,
        "script_feedback": (verdict.get("script_feedback") or "").strip(),
        "quiz_feedback": quiz_fb,
    }}


def node_render(state: QuizState) -> dict:
    return {"quiz_html": build_quiz_html(state["quiz_questions"], state.get("title", "Wissens-Check"))}

## Schritt 7 – Programmatische Qualitätschecks

Objektive, billige Prüfungen vor dem LLM-Urteil. Besonders wichtig: der **Wortzahl-Bias**.
LLMs neigen dazu, die korrekte Antwort am längsten/ausführlichsten zu formulieren – wir
messen das und erzwingen bei systematischem Bias eine Überarbeitung.

In [ ]:
def structural_checks(questions: List[dict], expected_n: int = 10) -> List[str]:
    issues = []
    if not isinstance(questions, list) or not questions:
        return ["Quiz enthält keine Fragen."]
    if len(questions) != expected_n:
        issues.append(f"Es wurden {len(questions)} statt {expected_n} Fragen erzeugt.")
    for i, q in enumerate(questions, 1):
        opts = q.get("options", [])
        if len(opts) != 4:
            issues.append(f"Frage {i}: {len(opts)} statt 4 Optionen.")
        c = q.get("correct")
        if not isinstance(c, int) or not (0 <= c <= 3):
            issues.append(f"Frage {i}: ungültiger correct-Index ({c}).")
    return issues


def word_balance_check(questions: List[dict], threshold: float = 0.4) -> List[str]:
    longest_is_correct, counted = 0, 0
    for q in questions:
        opts, c = q.get("options", []), q.get("correct")
        if len(opts) != 4 or not isinstance(c, int) or not (0 <= c <= 3):
            continue
        counted += 1
        lengths = [len(str(o).split()) for o in opts]
        if lengths.index(max(lengths)) == c and len(set(lengths)) > 1:
            longest_is_correct += 1
    if counted and longest_is_correct / counted > threshold:
        return [f"Wortzahl-Bias: bei {longest_is_correct}/{counted} Fragen ist die korrekte "
                f"Antwort die längste Option. Optionen angleichen."]
    return []

## Schritt 8 – Graph zusammenbauen und Routing

Zwei kompilierte Graphen aus denselben Knoten:

- **`full_graph`** – kompletter Lauf: `script -> termini -> quiz -> quality -> (render | zurück)`.
  Bei Skript-Mängeln geht es zurück zu `script` (Kaskade), bei reinen Quiz-Mängeln nur
  zu `quiz`. Nach **maximal 3 Iterationen** wird in jedem Fall gerendert (bestmögliches Ergebnis).
- **`quiz_graph`** – schlanker Pfad nur für neue Quizze: `quiz -> quality -> (render | quiz)`.
  Skript und Termini bleiben unangetastet.

In [ ]:
MAX_ITER = 3

def route_full(state: QuizState) -> str:
    q = state.get("quality", {})
    if q.get("script_ok", True) and q.get("quiz_ok", True):
        return "render"
    if state.get("iteration", 0) >= MAX_ITER:
        return "render"
    return "rewrite_script" if not q.get("script_ok", True) else "rewrite_quiz"


def route_quiz_only(state: QuizState) -> str:
    q = state.get("quality", {})
    if q.get("quiz_ok", True) or state.get("iteration", 0) >= MAX_ITER:
        return "render"
    return "rewrite_quiz"


def build_full_graph():
    g = StateGraph(QuizState)
    g.add_node("script", node_script)
    g.add_node("termini", node_termini)
    g.add_node("quiz", node_quiz)
    g.add_node("quality", node_quality)
    g.add_node("render", node_render)
    g.set_entry_point("script")
    g.add_edge("script", "termini")
    g.add_edge("termini", "quiz")
    g.add_edge("quiz", "quality")
    g.add_conditional_edges("quality", route_full,
                            {"render": "render", "rewrite_script": "script", "rewrite_quiz": "quiz"})
    g.add_edge("render", END)
    return g.compile()


def build_quiz_graph():
    g = StateGraph(QuizState)
    g.add_node("quiz", node_quiz)
    g.add_node("quality", node_quality)
    g.add_node("render", node_render)
    g.set_entry_point("quiz")
    g.add_edge("quiz", "quality")
    g.add_conditional_edges("quality", route_quiz_only,
                            {"render": "render", "rewrite_quiz": "quiz"})
    g.add_edge("render", END)
    return g.compile()


full_graph = build_full_graph()
quiz_graph = build_quiz_graph()

# Optionale Visualisierung des Graphen (benötigt zusätzliche Pakete):
try:
    from IPython.display import Image, display
    display(Image(full_graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Graph-Visualisierung übersprungen:", e)
    print(full_graph.get_graph().draw_mermaid())

## Schritt 9 – Gradio-Oberfläche

Oben die Modell-Einstellungen, darunter die zwei Eingaben und die Schwierigkeitsstufe.
Der **vollständige Lauf** erzeugt Skript, Termini und Quiz inklusive **Download-Dateien**.

Drei Komfort-Funktionen sind eingebaut:

- **Fortschrittsanzeige:** Über `graph.stream(...)` wird nach *jedem* Knoten der Fortschritt
  gemeldet (»Schritt-Label · Tokens bisher«), inklusive Korrektur-Durchläufen der
  Reflexions-Schleife. So sieht man bei langen DeepInfra-Läufen, dass es vorangeht.
  *(Hinweis: Die Meldung erscheint jeweils nach Abschluss eines Knotens – innerhalb eines
  einzelnen LLM-Aufrufs gibt es keine Zwischenschritte.)*
- **Token-Verbrauch:** Jeder Aufruf summiert die `usage`-Felder der Modellantwort. Nach
  jedem Lauf werden Prompt-, Antwort- und Gesamt-Tokens angezeigt (sofern der Endpunkt
  `usage` liefert – LM Studio, DeepInfra und OpenRouter tun das). Optional lässt sich ein
  Mischpreis pro 1 Mio. Tokens eingeben, um eine grobe Kostenschätzung zu erhalten.
- **Abwechslung bei »Neues Quiz«:** Damit kleine lokale Modelle nicht immer dieselben Fragen
  liefern, wird bei jeder Neu-Generierung ein **rotierender, zufälliger Satz Fachtermini**
  als synthetischer Schwerpunkt in den Prompt injiziert (plus leicht erhöhte Temperatur und
  die Liste bereits gestellter Fragen als Ausschluss). Das erzwingt inhaltlich andere Fragen
  statt minimaler Umformulierungen.

In [ ]:
SESSION = {}  # hält das Ergebnis des letzten vollständigen Laufs (Skript, Termini, Notebook ...)

# ---- Hilfsfunktionen -------------------------------------------------------
def _write_tmp(text: str, suffix: str) -> str:
    f = tempfile.NamedTemporaryFile("w", suffix=suffix, delete=False, encoding="utf-8")
    f.write(text); f.close()
    return f.name

def _preview_iframe(html_str: str) -> str:
    safe = html_lib.escape(html_str, quote=True)
    return f'<iframe srcdoc="{safe}" style="width:100%;height:600px;border:1px solid #ddd;border-radius:8px;"></iframe>'

def _fmt(n: int) -> str:
    return f"{n:,}".replace(",", ".")   # deutsche Tausendertrennung

def _token_md(price_per_m: float = 0.0, spotlight=None) -> str:
    t = TOKENS
    s = (f"**Token-Verbrauch (dieser Lauf):** {_fmt(t['total'])} gesamt · "
         f"{_fmt(t['prompt'])} Prompt · {_fmt(t['completion'])} Antwort · {t['calls']} LLM-Aufrufe")
    if price_per_m and t["total"]:
        cost = t["total"] / 1_000_000 * price_per_m
        s += f"\n\n**Grobe Kostenschätzung:** ≈ {cost:.4f} € (Mischpreis {price_per_m:.2f} €/1 Mio. Tokens)"
    if spotlight:
        s += "\n\n**Synthetischer Schwerpunkt dieses Quiz:** " + ", ".join(spotlight)
    return s

# Nominale Knotenreihenfolge (für den Fortschrittsanteil) und Klartext-Labels
FULL_ORDER = ["script", "termini", "quiz", "quality", "render"]
REGEN_ORDER = ["quiz", "quality", "render"]
STEP_LABELS = {"script": "Skript schreiben", "termini": "Fachtermini extrahieren",
               "quiz": "Quiz generieren", "quality": "Qualitätsprüfung", "render": "Fertigstellen"}

def _stream_with_progress(graph, state, order, progress):
    """Führt den Graphen aus und meldet nach jedem Knoten den Fortschritt."""
    total = len(order)
    final = dict(state)
    done = 0
    progress(0.0, desc="Modell wird angesprochen …")
    for update in graph.stream(state, stream_mode="updates"):
        for node, delta in update.items():
            if delta:
                final.update(delta)
            done += 1
            it = final.get("iteration", 0)
            extra = f" · Korrektur-Durchlauf {it}" if node in ("quiz", "quality") and it > 1 else ""
            frac = min(done / max(total, 1), 0.99)
            progress(frac, desc=f"{STEP_LABELS.get(node, node)}{extra} · {_fmt(TOKENS['total'])} Tokens bisher")
    progress(1.0, desc=f"Fertig · {_fmt(TOKENS['total'])} Tokens")
    return final

def _select_focus_termini(k: int = 5):
    """Wählt rotierend einen zufälligen Satz Fachtermini als synthetischen Schwerpunkt."""
    termini = list(SESSION.get("termini", []))
    if not termini:
        return []
    used = SESSION.get("used_focus_termini", [])
    pool = [t for t in termini if t not in used]
    if len(pool) < min(k, len(termini)):   # fast alle verbraucht -> Rotation zurücksetzen
        used, pool = [], termini[:]
    random.shuffle(pool)
    chosen = pool[:k]
    SESSION["used_focus_termini"] = used + chosen
    return chosen

# ---- Haupt-Handler ---------------------------------------------------------
def run_full(base_url, api_key, model, checker_base_url, checker_api_key, checker_model,
             nb_file, focus, difficulty, price_per_m, progress=gr.Progress()):
    if not base_url or not model:
        raise gr.Error("Bitte base_url und model angeben.")
    if nb_file is None:
        raise gr.Error("Bitte ein Jupyter-Notebook (.ipynb) hochladen.")
    reset_tokens()
    parsed = analyze_notebook(nb_file.name)
    state = {"base_url": base_url, "api_key": api_key, "model": model,
             "checker_base_url": checker_base_url or "", "checker_api_key": checker_api_key or "",
             "checker_model": checker_model or "",
             "focus": focus or "", "difficulty": int(difficulty),
             "avoid_topics": [], "quiz_focus_termini": [], "quiz_temperature": 0.7,
             "iteration": 0, "quality": {}, **parsed}
    result = _stream_with_progress(full_graph, state, FULL_ORDER, progress)

    termini_md = "## Fachtermini\n\n" + "\n".join(f"- {t}" for t in result.get("termini", []))
    SESSION.clear()
    SESSION.update({k: result[k] for k in ("base_url","api_key","model",
                                           "checker_base_url","checker_api_key","checker_model",
                                           "focus","difficulty","notebook_code","notebook_markdown",
                                           "script_md","title","termini") if k in result})
    SESSION["avoid_topics"] = [q.get("text","") for q in result.get("quiz_questions", [])]
    SESSION["used_focus_termini"] = []

    price = float(price_per_m or 0)
    return (result.get("script_md",""), termini_md, _preview_iframe(result["quiz_html"]),
            _write_tmp(result.get("script_md",""), "_skript.md"),
            _write_tmp(termini_md, "_termini.md"),
            _write_tmp(result["quiz_html"], "_quiz.html"),
            _token_md(price))

def run_regen(difficulty, price_per_m, progress=gr.Progress()):
    if "script_md" not in SESSION:
        raise gr.Error("Bitte zuerst einen vollständigen Lauf ausführen.")
    reset_tokens()
    spotlight = _select_focus_termini(5)
    state = {**SESSION, "difficulty": int(difficulty), "iteration": 0, "quality": {},
             "quiz_focus_termini": spotlight, "quiz_temperature": 0.9}
    result = _stream_with_progress(quiz_graph, state, REGEN_ORDER, progress)
    SESSION["avoid_topics"] = SESSION.get("avoid_topics", []) + [q.get("text","") for q in result.get("quiz_questions", [])]
    price = float(price_per_m or 0)
    return _preview_iframe(result["quiz_html"]), _write_tmp(result["quiz_html"], "_quiz.html"), _token_md(price, spotlight)

# ---- Oberfläche ------------------------------------------------------------
with gr.Blocks(title="Lernmaterial-Generator") as demo:
    gr.Markdown("# Lernmaterial-Generator\nNotebook + Schwerpunkte → Skript, Fachtermini, Quiz.")
    with gr.Row():
        base_url = gr.Textbox(label="base_url", value="http://localhost:1234/v1",
                              placeholder="LM Studio: http://localhost:1234/v1 · OpenRouter: https://openrouter.ai/api/v1")
        api_key = gr.Textbox(label="api_key", type="password", placeholder="bei LM Studio leer lassen")
        model = gr.Textbox(label="model", placeholder="z. B. meta-llama-3.1-8b-instruct")
    with gr.Accordion("Checker-Modell (optional – leer lassen = gleiches Modell wie oben)", open=False):
        gr.Markdown("Für die Qualitäts-/Halluzinationsprüfung lohnt oft ein stärkeres Modell. "
                    "Leere Felder fallen automatisch auf das Generator-Modell oben zurück; "
                    "`base_url`/`api_key` dürfen abweichen (anderer Provider möglich).")
        with gr.Row():
            checker_base_url = gr.Textbox(label="checker base_url", placeholder="leer = wie oben")
            checker_api_key = gr.Textbox(label="checker api_key", type="password", placeholder="leer = wie oben")
            checker_model = gr.Textbox(label="checker model", placeholder="z. B. ein stärkeres Modell für die Prüfung")
    with gr.Row():
        nb_file = gr.File(label="Jupyter-Notebook (.ipynb)", file_types=[".ipynb"])
        with gr.Column():
            focus = gr.Textbox(label="Schwerpunkte des Dozenten", lines=4,
                               placeholder="z. B. Nimm besonderen Bezug auf die Wahl des Optimizers und stelle die drei wichtigsten Optimizer gegenüber.")
            difficulty = gr.Slider(1, 5, value=3, step=1, label="Schwierigkeitsstufe (1–5)")
            price_per_m = gr.Number(value=0, label="Preis €/1 Mio. Tokens (optional, grobe Schätzung)")
    run_btn = gr.Button("Lernmaterial erzeugen", variant="primary")
    token_box = gr.Markdown()

    with gr.Tab("Skript"):
        out_script = gr.Markdown()
    with gr.Tab("Fachtermini"):
        out_termini = gr.Markdown()
    with gr.Tab("Quiz"):
        out_quiz = gr.HTML()
        with gr.Row():
            regen_diff = gr.Slider(1, 5, value=3, step=1, label="Schwierigkeit für neues Quiz")
            regen_btn = gr.Button("Neues Quiz erzeugen")
    with gr.Row():
        dl_script = gr.File(label="Skript herunterladen")
        dl_termini = gr.File(label="Termini herunterladen")
        dl_quiz = gr.File(label="Quiz herunterladen")

    run_btn.click(run_full, [base_url, api_key, model, checker_base_url, checker_api_key, checker_model,
                             nb_file, focus, difficulty, price_per_m],
                  [out_script, out_termini, out_quiz, dl_script, dl_termini, dl_quiz, token_box])
    regen_btn.click(run_regen, [regen_diff, price_per_m], [out_quiz, dl_quiz, token_box])

demo.launch(inbrowser=True)